# 02 – Join Layers to SWORD
**Purpose:** Expand SWORD reaches with attributes from vector and raster dataset. Each section adds columns to a growing SWORD GeoDataFrame. The final output contains all joined attributes.

**Workflow:**

2.0 | Imports

2.1 | Vector Data

- 2.1.1 | Line Join
- 2.1.2 | Point Join
- 2.1.3 | Polygon

2.2 | Raster Data
- 2.2.1 | STI glowabio

To-Do:
- [x] erstellte outputs aufeinander aufbauen lassen, sodass der SWORD Datensatz immer erweitert wird.
- [ ] Muss noch im notebook den SWORD datensatzaufbau einbauen und übersichtlicher gestalten
- [ ] sections einfügen, um zum SWORD hinzugefügte Datensätze mit dem original zu vergleichen
- [ ] Probleme mit Strahler Ordnung, da diese entweder zu fein ist (z.B. von GRIT strahler bis z.B. 113 für Naryn), es muss eher sowas wie eine connected components analyse sein...mit NEXT_DOWN von RiverATLAS?

- [] adding for connectivity dataset from: https://www.nature.com/articles/s41586-019-1111-9

---
---
## 2.0 | Imports

In [10]:
from shapely.geometry import Point
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import sys
import fiona
import matplotlib.pyplot as plt
import contextily as ctx
import webbrowser

sys.path.append(os.path.join("..", "src"))
from _0_config_paths import DATA_RAW, DATA_PROCESSED, DATA_OUTPUT
from _2_1_1_line import join_line_majority, compute_strahler_segments
from _2_1_2_point import snap_points_to_lines
from _2_2_1_raster import extract_raster_values, extract_raster_majority 

#### Input / Output cell:

In [13]:
# ============================================================
# STUDY AREA
# ============================================================
STUDY_AREA = "naryn"


# ============================================================
# INPUT / OUTPUT OVERVIEW
# ============================================================

#--- Base input (from Notebook 01) ---------------------------
IN_SWORD = os.path.join(DATA_PROCESSED, f"sword_{STUDY_AREA}_clip.gpkg")

# 2.1 LINE
#--- 2.1.1 gritv1.0 ----------------------------------------------
IN_GRIT = os.path.join(DATA_RAW, "0_data", f"{STUDY_AREA}_example", f"{STUDY_AREA}_gritv1.0.gpkg")
GRIT_LAYER = None

OUT_2_1_1 = os.path.join(DATA_PROCESSED, f"sword_{STUDY_AREA}_2_1_1.gpkg")
OUT_MAP_2_1_1 = os.path.join(DATA_OUTPUT, "02_1_1_grit_map.html")

#--- 2.1.2 riveratlas Line Join --------------------------------
IN_RIVERATLAS = os.path.join(DATA_RAW, "0_data", f"{STUDY_AREA}_example", 
                             f"{STUDY_AREA}_riveratlas_v10.gpkg")

# Columns to join from riveratlas
# NOTE: ORD_STRA replaces GRIT strahler_order as primary stream order
RIVERATLAS_COLS = [
    "ORD_STRA",    # Strahler stream order (classical, comparable across datasets)
    "ORD_CLAS",    # Shreve stream order
    "DIS_AV_CMS",  # mean annual discharge (m³/s)
    "UPLAND_SKM",  # upstream catchment area (km²)
    "slp_dg_cav",  # mean catchment slope (degrees)
    "ele_mt_cav",  # mean catchment elevation (m)
    "sgr_dk_rav",  # stream gradient (m/km)
    "lit_cl_cmj",  # dominant lithology class
    "HYRIV_ID",
    "NEXT_DOWN"
]

OUT_2_1_2 = os.path.join(DATA_PROCESSED, f"sword_{STUDY_AREA}_2_1_2_riveratlas.gpkg")
OUT_MAP_2_1_2 = os.path.join(DATA_OUTPUT, "02_1_2_riveratlas_map.html")

# 2.2 POINT
#--- 2.2.1 gdw barriers ---------------------------------------------
IN_GDW = os.path.join(DATA_RAW, "0_data", f"{STUDY_AREA}_example", f"{STUDY_AREA}_gdw_barriers_v1_0.gpkg")
OUT_2_2_1 = os.path.join(DATA_PROCESSED, f"sword_{STUDY_AREA}_2_2_1_gdw_snapped.gpkg")

# 2.3 POLYGON
#--- 2.3.1 Polygon Join: ... -------------------------


# 2.4 RASTER
#--- 2.4.1 STI glowabio ---------------------
RASTER_JOINS = [
    # (col_name, raster_path, method)
    # method: "continuous" --> mean/median | "categorical" --> majority vote
    
    ("STI_glowabio", os.path.join(DATA_RAW, "0_data", f"{STUDY_AREA}_example", "glowabio", f"{STUDY_AREA}_sti.tif"), "continuous"),
    
    # ESA worldcover – categorical (land cover classes)
    # NOTE: if multiple tiles were downloaded, they need to be merged first (see below)
    ("worldcover", os.path.join(DATA_RAW, "0_data", "worldcover", "worldcover_merged.tif"), "categorical"),
    
    # soilgrids – continuous (numeric soil properties)
    ("clay",  os.path.join(DATA_RAW, "0_data", "soilgrids", "clay_0-5cm_mean.tif"), "continuous"),
    ("sand",  os.path.join(DATA_RAW, "0_data", "soilgrids", "sand_0-5cm_mean.tif"), "continuous"),
    ("silt",  os.path.join(DATA_RAW, "0_data", "soilgrids", "silt_0-5cm_mean.tif"), "continuous"),
    ("cfvo",  os.path.join(DATA_RAW, "0_data", "soilgrids", "cfvo_0-5cm_mean.tif"), "continuous"),
]

OUT_2_4_1 = os.path.join(DATA_PROCESSED, f"sword_{STUDY_AREA}_2_4_1.gpkg")

#--- Final outputs --------------------------------------------------
OUT_FINAL     = os.path.join(DATA_PROCESSED, f"sword_{STUDY_AREA}_joined_final.gpkg")
OUT_MAP_FINAL = os.path.join(DATA_OUTPUT,    "02_final_map.html")

In [14]:
# Vertify all inputs exists:

check = [
    ("SWORD clipped", IN_SWORD),
    ("GRIT",IN_GRIT),
    ("GDW",IN_GDW),
    ("RiverATLAS", IN_RIVERATLAS),
] + [(f"Raster: {name} ({method})", path) for name, path, method in RASTER_JOINS]

all_ok = True
for label, path in check:
    exists = os.path.exists(path)
    status = "Found" if exists else "NOT FOUND: check path"
    print(f"{label:30s}: {status}")
    if not exists:
        all_ok = False

print()
print("All files found." if all_ok else "Fix missing paths.")

SWORD clipped                 : Found
GRIT                          : Found
GDW                           : Found
RiverATLAS                    : Found
Raster: STI_glowabio (continuous): Found
Raster: worldcover (categorical): Found
Raster: clay (continuous)     : Found
Raster: sand (continuous)     : Found
Raster: silt (continuous)     : Found
Raster: cfvo (continuous)     : Found

All files found.


---
---
## 2.1 | Vector Data

---
### 2.1.1 | Line Join – GRIT strahler order

Joins GRITv1.0 strahler order to each SWORD reach using majority-vote sampling. Sample points are placed every 50m along each reach. Each point is assigned to the nearest GRIT feature. The strahler order with the most votes wins.


<u>References:</u>

Wortmann, M., Slater, L., Hawker, L., Liu, Y., Neal, J., Zhang, B., et al. (2025) URL: https://aquaknow.jrc.ec.europa.eu/dataset/global-river-topology-grit

Wortmann, M., Slater, L., Hawker, L., Liu, Y., Neal, J., Zhang, B., et al. (2025). Global River Topology (GRIT): A bifurcating river hydrography. Water Resources Research, 61, e2024WR038308. https://doi.org/10.1029/2024WR038308


In [ ]:
# Load Data:
sword = gpd.read_file(IN_SWORD)
print(f"SWORD loaded: {len(sword)} reaches | CRS: {sword.crs}")

grit_layers = fiona.listlayers(IN_GRIT)
grit_layer = GRIT_LAYER if GRIT_LAYER else grit_layers[0]
grit_raw = gpd.read_file(IN_GRIT, layer=grit_layer)
print(f"GRIT loaded: {len(grit_raw)} features | CRS: {grit_raw.crs}")
print(f"GRIT columns: {grit_raw.columns.tolist()}")

Configuration Cell:

In [ ]:
# Configuration:
# Which cols should be joined
GRIT_COLS_TO_JOIN = ["strahler_order", "global_id", "geometry"]
# Rename added columns in SWORD dataset
GRIT_RENAME= {
    "strahler_order" : "strahler_order_GRITv1.0",
    "global_id" : "global_id_GRITv1.0"
}
MAX_DISTANCE_M = 100
SAMPLE_DISTANCE_M = 50

# Prepare GRIT: keep only needed columns
missing = [c for c in GRIT_COLS_TO_JOIN
           if c != "geometry" and c not in grit_raw.columns]
if missing:
    print(f"Columns not found in GRIT: {missing}")
else:
    print("All requested GRIT columns found.")

cols_keep = [c for c in GRIT_COLS_TO_JOIN if c in grit_raw.columns]
grit = grit_raw[cols_keep].copy()

In [ ]:
# Function from "_2_1_1_line.py"
sword_2_1_1 = join_line_majority(
    sword= sword,
    grit= grit,
    cols_to_join = GRIT_COLS_TO_JOIN,
    rename_cols= GRIT_RENAME,
    max_distance_m = MAX_DISTANCE_M,
    sample_distance_m = SAMPLE_DISTANCE_M,
    prefix="grit"
)

# Convert to integer
for col in ["strahler_order_GRITv1.0", "global_id_GRITv1.0"]:
    if col in sword_2_1_1.columns:
        sword_2_1_1[col] = sword_2_1_1[col].astype("Int64")

print("\nNew columns added:")
new_cols = ["strahler_order_GRITv1.0", "global_id_GRITv1.0", "grit_majority_confidence"]
print(sword_2_1_1[["reach_id"] + new_cols].head(10))

In [ ]:
# Quality check:
print("Majority confidence score [0-1]:")
print(sword_2_1_1["grit_majority_confidence"].describe().round(3))

print(f"Ambiguous reaches (confidence < 0.6 or NaN): {sword_2_1_1['grit_ambiguous'].sum()}")

print("\nReaches per strahler_order_GRITv1.0:")
print(sword_2_1_1["strahler_order_GRITv1.0"].value_counts().sort_index())

# Save
os.makedirs(DATA_PROCESSED, exist_ok=True)
sword_2_1_1.to_file(OUT_2_1_1, driver="GPKG")
print(f"\nSaved: {OUT_2_1_1}")

LATEST = OUT_2_1_1

In [ ]:
# Interactive Map to check the joined strahler order information:
m = sword_2_1_1.explore(
    column="reach_id",
    cmap="turbo",
    tooltip=["reach_id", "river_name", "strahler_order_GRITv1.0",
             "strm_order", "grit_majority_confidence", "global_id_GRITv1.0"],
    tiles="CartoDB positron",
    legend=True
)
m.save(OUT_MAP_2_1_1)
webbrowser.open(OUT_MAP_2_1_1)
print(f"Map saved: {OUT_MAP_2_1_1}")

Visual comparison between original SWORD reaches and to GRITv1.0 strahler order classified SWORD reaches.

In [ ]:
# ################################################################
# ### VISUAL COMPARISON ###
# #########################

# # creating two plots
# fig, axes = plt.subplots(3, 1, figsize=(10, 20))

# # Reproject to metric CRS for display
# sword_plot = sword_2_1_1.to_crs("EPSG:32643")

# #Left map: SWORD column "strm_order"
# sword_plot.plot(
#     ax=axes[0],
#     column="reach_id",
#     cmap="Set1",
#     linewidth=0.8,
#     legend=True,
#     legend_kwds={"label": "SWORD strm_order", "shrink": 0.6}
# )
# axes[0].set_title("SWORD reaches\n(original)", fontsize=12)
# axes[0].axis("off")

# #Left map: SWORD column "strm_order"
# sword_plot.plot(
#     ax=axes[1],
#     column="strm_order",
#     cmap="Set1",
#     linewidth=0.8,
#     legend=True,
#     legend_kwds={"label": "SWORD strm_order", "shrink": 0.6}
# )
# axes[0].set_title("SWORD reaches\n(original)", fontsize=12)
# axes[0].axis("off")

# # Right map: the added GRIT strahler_order (column: "strahler_order_GRITv1.0")
# sword_plot.plot(
#     ax=axes[2],
#     column="strahler_order_GRITv1.0",
#     cmap="turbo",
#     linewidth=0.8,
#     legend=True,
#     legend_kwds={"label": "GRIT strahler_order", "shrink": 0.6}
# )
# axes[1].set_title("GRIT: strahler_order_GRITv1.0\n(joined via majority vote)", fontsize=12)
# axes[1].axis("off")

# plt.suptitle("Comparison: SWORD strm_order vs. GRIT strahler_order", fontsize=14)
# plt.tight_layout()

# # Save
# out_comparison = os.path.join(DATA_OUTPUT, "02_1_1_comparison_strm_vs_grit.png")
# plt.savefig(out_comparison, dpi=150, bbox_inches="tight")
# plt.show()
# print(f"Saved: {out_comparison}")

# which SWORD reaches share the same GRIT strahler order segment 

grouped = (sword_2_1_1
    .dropna(subset=["global_id_GRITv1.0"])
    .groupby("global_id_GRITv1.0")
    .agg(
        strahler_order = ("strahler_order_GRITv1.0", "first"),
        n_reaches= ("reach_id", "count"),
        reach_ids = ("reach_id", list),
        confidence_min = ("grit_majority_confidence", "min"),
        confidence_mean= ("grit_majority_confidence", "mean")
    )
    .sort_values("n_reaches", ascending=False)
    .reset_index()
)

print(f"Total GRIT segments (global_ids): {len(grouped)}")
print(f"SWORD reaches covered: {grouped['n_reaches'].sum()}")
print(f"\nSegments with only 1 reach: {(grouped['n_reaches'] == 1).sum()}")
print(f"Segments with 2-5 reaches: {grouped['n_reaches'].between(2,5).sum()}")
print(f"Segments with >5 reaches: {(grouped['n_reaches'] > 5).sum()}")

print("\n" + "-"*70)
print("GRIT segments with multiple SWORD reaches:")


for _, row in grouped[grouped["n_reaches"] > 1].iterrows():
    print(f"\nglobal_id : {int(row['global_id_GRITv1.0']):<10} "
          f"strahler: {int(row['strahler_order']):<5} "
          f"n_reaches: {int(row['n_reaches']):<5} "
          f"conf_mean: {row['confidence_mean']:.2f}")
    print(f"  reach_ids: {row['reach_ids']}")
print("\n" + "-"*70)

grouped[["global_id_GRITv1.0", "strahler_order",
         "n_reaches", "confidence_min", "confidence_mean"]]

### 2.1.2 | RiverATLAS

Adding strahler order and some other variables from RiverATLAS v.1.0 instead of GRIT...because propably better usage for the egg.

In [ ]:
# Load Data:
# Load Data:
sword = gpd.read_file(OUT_2_1_1)  # load SWORD with GRIT join from previous section
print(f"SWORD loaded: {len(sword)} reaches | CRS: {sword.crs}")

riveratlas_raw = gpd.read_file(IN_RIVERATLAS)
print(f"RiverATLAS loaded: {len(riveratlas_raw)} features | CRS: {riveratlas_raw.crs}")
print(f"RiverATLAS columns: {riveratlas_raw.columns.tolist()}")

# Configuration:
# Which cols should be joined
# Rename added columns in SWORD dataset
GRIT_RENAME = {
    "ORD_STRA": "strahler_order_RiverATLAS",
    "ORD_CLAS": "shreve_stream_order_RiverATLAS",   # Shreve stream order
    "DIS_AV_CMS": "dis_RiverATLAS", # mean annual discharge (m³/s)
    "UPLAND_SKM": "upland_skm_RiverATLAS",  # upstream catchment area (km²)
    "slp_dg_cav": "slp_dg_cav_RiverATLAS",  # mean catchment slope (degrees)
    "ele_mt_cav": "ele_mt_cav_RiverATLAS",  # mean catchment elevation (m)
    "sgr_dk_rav": "stream_gradient_RiverATLAS",  # stream gradient (m/km)
    "lit_cl_cmj": "dom_lithology_RiverATLAS",
    "HYRIV_ID"  : "HYRIV_ID_RiverATLAS",  # dominant lithology class
    "strahler_segment_id": "strahler_segment_id_RiverATLAS",
    "NEXT_DOWN": "NEXT_DOWN_RiverATLAS",
}


# Compute spatially connected Strahler segments from RiverATLAS topology
riveratlas_segmented = compute_strahler_segments(riveratlas_raw)

# Add strahler_segment_id to RIVERATLAS_COLS so it gets joined to SWORD
RIVERATLAS_COLS_WITH_SEG = RIVERATLAS_COLS + ["strahler_segment_id"]

MAX_DISTANCE_M = 100
SAMPLE_DISTANCE_M = 50

# Prepare GRIT: keep only needed columns
missing = [c for c in RIVERATLAS_COLS
           if c != "geometry" and c not in riveratlas_segmented.columns]
if missing:
    print(f"Columns not found in RiverATLAS: {missing}")
else:
    print("All requested GRIT columns found.")

# Function from "_2_1_1_line.py"
sword_2_1_2 = join_line_majority(
    sword= sword,
    grit= riveratlas_segmented[RIVERATLAS_COLS_WITH_SEG + ["geometry"]],
    cols_to_join = RIVERATLAS_COLS_WITH_SEG,
    rename_cols= GRIT_RENAME,
    max_distance_m = MAX_DISTANCE_M,
    sample_distance_m = SAMPLE_DISTANCE_M,
    prefix="RiverATLAS"
)

# Integer columns:
for col in ["strahler_order_RiverATLAS",
            "shreve_stream_order_RiverATLAS",
            "dom_lithology_RiverATLAS"]:
    if col in sword_2_1_2.columns:
        sword_2_1_2[col] = sword_2_1_2[col].astype("Int64")

# Float columns – keep as float, just round for readability:
for col in ["dis_RiverATLAS", "upland_skm_RiverATLAS",
            "slp_dg_cav_RiverATLAS", "ele_mt_cav_RiverATLAS",
            "stream_gradient_RiverATLAS"]:
    if col in sword_2_1_2.columns:
        sword_2_1_2[col] = sword_2_1_2[col].round(3)

print("\nNew columns added:")
new_cols = ["strahler_order_RiverATLAS",
    "shreve_stream_order_RiverATLAS",
    "dis_RiverATLAS",
    "upland_skm_RiverATLAS",
    "slp_dg_cav_RiverATLAS",
    "ele_mt_cav_RiverATLAS",
    "stream_gradient_RiverATLAS",
    "dom_lithology_RiverATLAS",
    "HYRIV_ID_RiverATLAS", 
    "RiverATLAS_majority_confidence",
    ]
print(sword_2_1_2[["reach_id"] + new_cols].head(10))

# Quality check:
print("\nUnique HYRIV_ID groups (= potential eggs):")
print(sword_2_1_2["HYRIV_ID_RiverATLAS"].nunique())
print(f"Avg SWORD reaches per HYRIV_ID: {len(sword_2_1_2) / sword_2_1_2['HYRIV_ID_RiverATLAS'].nunique():.1f}")

print("Majority confidence score [0-1]:")
print(sword_2_1_2["RiverATLAS_majority_confidence"].describe().round(3))

print(f"Ambiguous reaches (confidence < 0.6 or NaN): {sword_2_1_2['RiverATLAS_ambiguous'].sum()}")

print("\nReaches per strahler_order_RiverATLAS:")
print(sword_2_1_2["strahler_order_RiverATLAS"].value_counts().sort_index())

# Save
os.makedirs(DATA_PROCESSED, exist_ok=True)
sword_2_1_2.to_file(OUT_2_1_2, driver="GPKG")
print(f"\nSaved: {OUT_2_1_2}")

LATEST = OUT_2_1_2

# Interactive Map to check the joined strahler order information:
m = sword_2_1_2.explore(
    column="reach_id",
    cmap="turbo",
    tooltip=["reach_id", "river_name", "strahler_order_RiverATLAS","RiverATLAS_majority_confidence"],
    tiles="CartoDB positron",
    legend=True
)
m.save(OUT_MAP_2_1_2)
webbrowser.open(OUT_MAP_2_1_2)
print(f"Map saved: {OUT_MAP_2_1_2}")

---
## 2.2 | Point Snap 

---
### 2.2.1 | GDW barrier

Snaps GDW barrier points to the nearest SWORD reach line. Only instream features are used. Snapped points are exported separately. The snap result (reach_id) is later used to flag reaches with upstream dams.

To-Do:
- [ ] Some of the original GDW points are located quite far from the feature, which is why the spatial deviation from the dam remains quite high even after snapping.

References:

Ward, B. (2019): "How to leverage Geopandas for faster snapping of points to lines". URL: https://medium.com/@brendan_ward/how-to-leverage-geopandas-for-faster-snapping-of-points-to-lines-6113c94e59aa

Lehner, B. et al.(2024): Global Dam Watch database version 1.0. URL: https://figshare.com/articles/dataset/Global_Dam_Watch_database_version_1_0/25988293

https://www.globaldamwatch.org/database

In [ ]:
# Reload SWORD from 2.1.1 output:
sword_2_1_1 = gpd.read_file(OUT_2_1_1)
print(f"SWORD reloaded: {len(sword_2_1_1)} reaches")

gdw = gpd.read_file(IN_GDW)
print(f"GDW loaded: {len(gdw)} features | CRS: {gdw.crs}")
print(f"GDW columns: {gdw.columns.tolist()}")

# Filtering for instream features only:
gdw = gdw[gdw["INSTREAM"].str.lower() == "instream"]
print(f"GDW instream: {len(gdw)} features after filter")

In [ ]:
# Configuration:
# The tolerance in meter describes the max snap distance
SNAP_TOLERANCE_M = 164

# NOTE: GDW points are exported separately, not merged into SWORD.
# The reach_id column links snapped points back to SWORD reaches.

In [ ]:
# Function from src file "_2_1_2_point.py"
snapped_gdw = snap_points_to_lines(
    points = gdw,
    lines = sword_2_1_1,
    tolerance_m = SNAP_TOLERANCE_M,
    line_id_col = "reach_id"
)

# Save snapped points (separate file -> not merged into SWORD)
snapped_gdw.to_file(OUT_2_2_1, driver="GPKG")
print(f"Saved: {OUT_2_2_1}")

POINTS = OUT_2_2_1

---
## 2.3 | Polygon 

...

---
---
## 2.4 | Raster Data

---
### 2.4.1 | Raster Join: STI glowabio

Extracts raster pixel values along each SWORD reach. Buffer per reach = (width/2) + offset to account for SWORD positional uncertainty (~50–200m offset from actual river).


Since the reaches can be offset by up to 200 meters from the actual river, a buffer is set around the reaches,depending on the reach width (column: "width"), to extract pixels from raster data. 

To-Do:

- [ ] I don't like the buffer solution because, with the current logic, it could end up extracting an unnecessary number of incorrect pixels. Maybe I should use a flow direction grid to better specify the direction of the buffer?
- [ ] In "raster_join.py" add to function "def extract_raster_values" that only pixels which overlap the buffer with a min. area of.... =< 50%? are included in the mean/median calculation.

https://glowabio.org/

https://hydrography.org/hydrography90m/hydrography90m_layers/


In [ ]:
# Reload from latest vector output
sword_current = gpd.read_file(LATEST)
print(f"SWORD reloaded: {len(sword_current)} reaches")

sword_2_4_1 = sword_current.copy()


for col_name, raster_path, method in RASTER_JOINS:
    print(f"\nRaster: {raster_path}")
    print(f"Method: {method}")
    
    if method == "continuous":
        sword_2_4_1 = extract_raster_values(
            sword_2_4_1, raster_path, col_name, width_col="width"
        )
    elif method == "categorical":
        sword_2_4_1 = extract_raster_majority(
            sword_2_4_1, raster_path, col_name, width_col="width"
        )
    else:
        print(f"Unknown method '{method}' for {col_name} - skipping")

In [ ]:
# Inspecting results, build column list based on method
raster_cols = []
for name, _, method in RASTER_JOINS:
    if method == "continuous":
        raster_cols += [f"{name}_mean", f"{name}_median"]
    elif method == "categorical":
        raster_cols += [f"{name}_majority", f"{name}_majority_confidence"]

# Filter to only existing columns
raster_cols = [c for c in raster_cols if c in sword_2_4_1.columns]

print("Sample:")
print(sword_2_4_1[["reach_id", "width"] + raster_cols].head(10))

print("\nStatistics:")
print(sword_2_4_1[raster_cols].describe())

# Save
sword_2_4_1.to_file(OUT_2_4_1, driver="GPKG")
print(f"\nSaved: {OUT_2_4_1}")

LATEST = OUT_2_4_1

In [ ]:
print(sword_2_4_1.columns.tolist())

---
---
## 2.5 | FINAL OUTPUT

Combining all joined attributes into one final SWORD GeoDataFrame.
This is the input for Notebook 03 (classification).

In [ ]:
# Load the latest raster-joined dataset as base
sword_final = gpd.read_file(LATEST)

print(f"Final SWORD dataset:")
print(f"Reaches : {len(sword_final)}")
print(f"Columns : {sword_final.columns.tolist()}")

sword_final.to_file(OUT_FINAL, driver="GPKG")
print(f"\nSaved final output: {OUT_FINAL}")

In [ ]:
tooltip_cols = [c for c in [
    "reach_id", "river_name", "strahler_order_GRITv1.0",
    "grit_majority_confidence", "RiverATLAS_majority_confidence"
] if c in sword_final.columns]

m = sword_final.explore(
    column="strahler_order_GRITv1.0",
    cmap="RdYlBu",
    tooltip=tooltip_cols,
    tiles="CartoDB positron",
    legend=True
)
m.save(OUT_MAP_FINAL)
webbrowser.open(OUT_MAP_FINAL)
print(f"Map saved: {OUT_MAP_FINAL}")